In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# -------------------------
# Load TEA-seq Metadata
# -------------------------
# Assuming the file generated by your R script
metadata_path = "../data/cleaned_cell_labels_meta_tea_seq.csv"
if not Path(metadata_path).exists():
    raise FileNotFoundError(f"Metadata not found at {metadata_path}. Please run the R processing script first.")

mdata = pd.read_csv(metadata_path)

# In your R script, we saved columns as 'CellID' and 'CellType' (or it might be the index)
# Adjusting to ensure we grab the labels correctly
if "CellType" in mdata.columns:
    y = mdata["CellType"].to_numpy()
else:
    # Fallback if the CSV structure varies
    y = mdata.iloc[:, 1].to_numpy()

n = len(y)
idx_all = np.arange(n)

In [ ]:
# Make sure splits directory exists
Path("../splits").mkdir(exist_ok=True, parents=True)

In [ ]:
# -------------------------
# Define split specifications (TEA-seq)
# -------------------------

split_specs = {
    # Split 3: all cell types
    "split3_all_celltypes": {
        "allowed_celltypes": None  # no restriction
    },
}

# -------------------------
# Helper to create and save Hyperparameter split
# -------------------------

def make_and_save_hyper_split(split_name, allowed_celltypes):
    # Filter the population based on allowed cell types
    if allowed_celltypes is None:
        mask = np.ones_like(y, dtype=bool)
    else:
        mask = np.isin(y, list(allowed_celltypes))

    idx_subset = idx_all[mask]
    y_subset = y[mask]

    # Check if we have enough cells to split
    if idx_subset.size < 10:
        print(f"Skipping {split_name}: too few cells ({idx_subset.size})")
        return

    print(f"Processing {split_name}: {idx_subset.size} cells total...")

    # Select 10% for Hyperparameter Estimation
    # Stratify=y_subset ensures the 10% has the same cell type proportions as the full subset
    _, idx_hyper_sub = train_test_split(
        idx_subset,
        test_size=0.10,
        random_state=42,
        stratify=y_subset,
    )

    # Save to CSV
    output_df = pd.DataFrame({"index": idx_hyper_sub})
    output_path = f"../splits/tea_{split_name}_hyper_idx.csv"
    output_df.to_csv(output_path, index=False)

    print(f"  --> Saved {len(idx_hyper_sub)} indices to {output_path}")

# -------------------------
# Main Loop
# -------------------------

for split_name, conf in split_specs.items():
    make_and_save_hyper_split(split_name, conf["allowed_celltypes"])

print("\nAll hyperparameter splits generated successfully.")